In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-openai sentence-transformers transformers chromadb panel param


In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


# Chat

In [ ]:
import panel as pn
pn.extension()

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from google.colab import drive
import torch

drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)


### Memory + ConversationalRetrievalChain

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

SYSTEM_TEMPLATE = """انت مساعد قانوني متخصص في القانون البحريني. استخدم المقاطع القانونية التالية فقط للاجابة على السؤال في نهاية النص.

قواعد صارمة يجب اتباعها:
- استند فقط الى النصوص المرفقة، ولا تخترع اي معلومة غير موجودة فيها.
- اذا لم تكن الاجابة موجودة في النصوص المرفقة، صرح بذلك بوضوح ولا تخمن.
- اذكر المصدر الدقيق لكل معلومة (رقم المادة او رقم القضية).

النصوص القانونية:
{context}

السؤال: {question}

الاجابة القانونية المدعومة بالمصادر:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(SYSTEM_TEMPLATE)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2})

qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT},
)

In [ ]:
question = "هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون انذار؟"
result = qa.invoke({"question": question})
result["answer"]

In [ ]:
follow_up = "ما هي المدة التي يجب ان يغيبها العامل حتى يجوز فصله؟"
result = qa.invoke({"question": follow_up})
result["answer"]

## Create a chatbot that works on our legal documents

In [ ]:
import param

def load_qa_chain(llm_model="nvidia/nemotron-3-ultra-550b-a55b:free"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
    vectordb = Chroma(persist_directory="/content/drive/MyDrive/law_chatbot_chroma_v2", embedding_function=embedding)
    llm = ChatOpenAI(
        model=llm_model,
        temperature=0,
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    return ConversationalRetrievalChain.from_llm(
        llm,
        retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2}),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT},
    )


class cbfs(param.Parameterized):
    chat_history = param.List([])
    answer = param.String("")
    db_response = param.List([])

    def __init__(self, **params):
        super(cbfs, self).__init__(**params)
        self.panels = []
        self.qa = load_qa_chain()

    def convchain(self, query):
        if not query:
            return pn.WidgetBox(pn.Row("المستخدم:", pn.pane.Markdown("", width=600)), scroll=True)
        result = self.qa.invoke({"question": query})
        self.chat_history.extend([(query, result["answer"])])
        self.db_response = result.get("source_documents", [])
        self.answer = result["answer"]
        self.panels.extend([
            pn.Row("المستخدم:", pn.pane.Markdown(query, width=600)),
            pn.Row("المساعد القانوني:", pn.pane.Markdown(self.answer, width=600, styles={"background-color": "#F6F6F6"})),
        ])
        inp.value = ""
        return pn.WidgetBox(*self.panels, scroll=True)

    @param.depends("db_response")
    def get_sources(self):
        if not self.db_response:
            return
        rlist = [pn.Row(pn.pane.Markdown("المصادر المسترجعة:", styles={"background-color": "#F6F6F6"}))]
        for doc in self.db_response:
            rlist.append(pn.Row(pn.pane.Str(f"{doc.metadata} — {doc.page_content[:150]}")))
        return pn.WidgetBox(*rlist, width=600, scroll=True)

    def clr_history(self, count=0):
        self.chat_history = []
        self.panels = []
        return

### Dashboard

In [ ]:
cb = cbfs()

button_clearhistory = pn.widgets.Button(name="مسح المحادثة", button_type="warning")
button_clearhistory.on_click(cb.clr_history)
inp = pn.widgets.TextInput(placeholder="اكتب سؤالك القانوني هنا...")

conversation = pn.bind(cb.convchain, inp)

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation, loading_indicator=True, height=400),
)
tab2 = pn.Column(pn.panel(cb.get_sources))
tab3 = pn.Column(
    pn.Row(button_clearhistory, pn.pane.Markdown("يمسح سجل المحادثة لبدء موضوع جديد")),
)

dashboard = pn.Column(
    pn.Row(pn.pane.Markdown("# المساعد القانوني - Capital Legal Base")),
    pn.Tabs(("المحادثة", tab1), ("المصادر", tab2), ("الإعدادات", tab3)),
)
dashboard

### Note on the demo environment